In [5]:
import pandas as pd
import numpy as np


In [6]:
df = pd.read_csv(r'data_center\Retail_Prices_of_Products.csv', encoding='iso-8859-1', on_bad_lines='skip')

df.info()

df.head(-1).T

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 118482 entries, 0 to 118481
Data columns (total 12 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   Year              118482 non-null  int64  
 1   Month             118482 non-null  object 
 2   GEO               118482 non-null  object 
 3   Product Category  118482 non-null  object 
 4   Products          118482 non-null  object 
 5   VALUE             118482 non-null  float64
 6   Taxable           118482 non-null  object 
 7   Total tax rate    118482 non-null  float64
 8   Value after tax   118482 non-null  float64
 9   Essential         118482 non-null  object 
 10  COORDINATE        118482 non-null  float64
 11  UOM               118482 non-null  object 
dtypes: float64(4), int64(1), object(7)
memory usage: 10.8+ MB


,0,1,2,3,4,5,6,7,8,9,...,118471,118472,118473,118474,118475,118476,118477,118478,118479,118480
Year,2017,2017,2017,2017,2017,2017,2017,2017,2017,2017,...,2025,2025,2025,2025,2025,2025,2025,2025,2025,2025
Month,January,January,January,January,January,January,January,January,January,January,...,February,February,February,February,February,February,February,February,February,February
GEO,Province 1,Province 1,Province 1,Province 1,Province 1,Province 1,Province 1,Province 1,Province 1,Province 1,...,Province 11,Province 11,Province 11,Province 11,Province 11,Province 11,Province 11,Province 11,Province 11,Province 11
Product Category,Meat & Poultry,Meat & Poultry,Meat & Poultry,Meat & Poultry,Meat & Poultry,Meat & Poultry,Meat & Poultry,Meat & Poultry,Meat & Poultry,Meat & Poultry,...,Legumes & Dry Goods,Canned & Jarred Goods,Grains & Bakery,Canned & Jarred Goods,Nuts & Snacks,Nuts & Snacks,Nuts & Snacks,Toiletries & Cleaning,Toiletries & Cleaning,Toiletries & Cleaning
Products,"Beef stewing cuts, per kilogram","Beef striploin cuts, per kilogram","Beef top sirloin cuts, per kilogram","Beef rib cuts, per kilogram","Ground beef, per kilogram","Pork loin cuts, per kilogram","Pork rib cuts, per kilogram","Pork shoulder cuts, per kilogram","Whole chicken, per kilogram","Chicken breasts, per kilogram",...,"Hummus, 227 grams","Salsa, 418 millilitres","Pasta sauce, 650 millilitres","Salad dressing, 475 millilitres","Almonds, 200 grams","Peanuts, 450 grams","Sunflower seeds, 400 grams","Deodorant, 85 grams","Toothpaste, 100 millilitres","Shampoo, 400 millilitres"
VALUE,12.66,21.94,13.44,20.17,9.12,7.34,7.37,4.76,5.15,11.38,...,3.82,4.53,3.1,3.44,4.52,4.05,4.5,8.07,4.13,7.18
Taxable,No,No,No,No,No,No,No,No,No,No,...,No,Yes,No,Yes,No,No,No,Yes,Yes,Yes
Total tax rate,11.0,11.0,11.0,11.0,11.0,11.0,11.0,11.0,11.0,11.0,...,12.0,12.0,12.0,12.0,12.0,12.0,12.0,12.0,12.0,12.0
Value after tax,12.66,21.94,13.44,20.17,9.12,7.34,7.37,4.76,5.15,11.38,...,3.82,5.07,3.1,3.85,4.52,4.05,4.5,9.04,4.63,8.04
Essential,Essential,Essential,Essential,Essential,Essential,Essential,Essential,Essential,Essential,Essential,...,Essential,Non-Essential,Essential,Non-Essential,Non-Essential,Non-Essential,Non-Essential,Essential,Essential,Essential


In [51]:

quality_keywords = {
    'striploin': 9,
    'rib': 8,
    'sirloin': 7,
    'stewing': 6,
    'ground': 5,
    'chicken': 5,
    'turkey': 6,
    'duck': 7,
    'pork': 6,
    'lamb': 8,
    'veal': 9,
    'organ': 3,
    'sausages': 4,
}
def estimate_quality(product_name):
    for key, val in quality_keywords.items():
        if key.lower() in product_name.lower():
            return val
    return 5

df['QualityScore'] = df['Products'].apply(estimate_quality)


df['NormPrice'] = (df['VALUE'] - df['VALUE'].mean()) / df['VALUE'].std()


df['Essential_Bonus'] = df['Essential'].apply(lambda x: 1 if x == 'Non-Essential' else 0)


region_weights = {
    'Province 1': 0.2,
    'Province 2': -0.3,
    'Province 3': 0.5,
    'Province 4': 0.0,
    'Province 5': -0.1,
    'Province 6': 0.4,
    'Province 7': -0.2,
    'Province 8': 0.3,
    'Province 9': -0.4,
    'Province 10': 0.1,
}
df['Region_Weight'] = df['GEO'].map(region_weights).fillna(0)

month_weights = {
    'January': -0.2,
    'February': 0.0,
    'March': -0.1,
    'April': 0.0,
    'May': 0.2,
    'June': 0.3,
    'July': 0.3,
    'August': 0.2,
    'September': 0.1,
    'October': 0.0,
    'November': 0.3,
    'December': 0.6,
}
df['Month_Weight'] = df['Month'].map(month_weights).fillna(0)


df['Year_Trend'] = (df['Year'] - df['Year'].min()) * 0.1


np.random.seed(42)
df['Review'] = (
    df['QualityScore'] * 0.5 +
    df['NormPrice'] * 0.8 +
    df['Essential_Bonus'] * 1.0 +
    df['Region_Weight'] * 1.5 +
    df['Month_Weight'] * 1.0 +
    df['Year_Trend'] * 1.2 +
    np.random.normal(0, 0.8, size=len(df))
)


df['Review'] = df['Review'].clip(1, 10).round(1)


outlier_indices = np.random.choice(df.index, size=10, replace=False)
df.loc[outlier_indices, 'Review'] = np.random.choice([1.0, 10.0], size=10)


df.to_csv("Retail_With_Review.csv", index=False)

print(df[['Year', 'Month', 'Products', 'VALUE', 'Essential', 'GEO', 'Review']].head())

   Year    Month                             Products  VALUE  Essential  \
0  2017  January      Beef stewing cuts, per kilogram  12.66  Essential   
1  2017  January    Beef striploin cuts, per kilogram  21.94  Essential   
2  2017  January  Beef top sirloin cuts, per kilogram  13.44  Essential   
3  2017  January          Beef rib cuts, per kilogram  20.17  Essential   
4  2017  January            Ground beef, per kilogram   9.12  Essential   

          GEO  Review  
0  Province 1     4.6  
1  Province 1     7.0  
2  Province 1     5.4  
3  Province 1     7.6  
4  Province 1     3.0  


In [40]:
unique_counts = df['Products'].nunique()
unique_values = df['Products'].unique()
print(unique_counts)
print(unique_values)

110
['Beef stewing cuts, per kilogram' 'Beef striploin cuts, per kilogram'
 'Beef top sirloin cuts, per kilogram' 'Beef rib cuts, per kilogram'
 'Ground beef, per kilogram' 'Pork loin cuts, per kilogram'
 'Pork rib cuts, per kilogram' 'Pork shoulder cuts, per kilogram'
 'Whole chicken, per kilogram' 'Chicken breasts, per kilogram'
 'Chicken thigh, per kilogram' 'Chicken drumsticks, per kilogram'
 'Bacon, 500 grams' 'Wieners, 400 grams' 'Salmon, per kilogram'
 'Shrimp, 300 grams' 'Canned salmon, 213 grams' 'Canned tuna, 170 grams'
 'Meatless burgers, 226 grams' 'Milk, 1 litre' 'Milk, 2 litres'
 'Milk, 4 litres' 'Soy milk, 1.89 litres' 'Nut milk, 1.89 litres'
 'Cream, 1 litre' 'Butter, 454 grams' 'Margarine, 907 grams'
 'Block cheese, 500 grams' 'Yogurt, 500 grams' 'Eggs, 1 dozen'
 'Apples, per kilogram' 'Oranges, per kilogram' 'Oranges, 1.36 kilograms'
 'Bananas, per kilogram' 'Pears, per kilogram' 'Lemons, unit'
 'Limes, unit' 'Grapes, per kilogram' 'Cantaloupe, unit'
 'Strawberries, 4

In [52]:
df.to_csv('Retail_Prices_of_Products_M.csv', index=False, encoding='utf-8')